# Tabular Prediction with AutoGluon

AutoGluon's `TabularPredictor` automates the machine learning pipeline for structured (tabular) data. It trains dozens of models, tunes hyperparameters, and builds a weighted ensemble — all with a single `.fit()` call.

This notebook covers:
1. Binary classification (Adult income: will this person earn >50K/year?)
2. Reading the leaderboard and understanding model performance
3. Making predictions and computing feature importance
4. A quick regression demo (house prices)
5. Practical gotchas to watch for

In [ ]:
import autogluon
print('AutoGluon version:', autogluon.__version__)

## 1. Load Data

`TabularDataset` is just a pandas DataFrame with a convenience loader. You can use any DataFrame — it is not required.

In [ ]:
from autogluon.tabular import TabularDataset, TabularPredictor

train_data = TabularDataset('https://autogluon.s3.amazonaws.com/datasets/AdultIncomeBinaryClassification/train_data.csv')
test_data  = TabularDataset('https://autogluon.s3.amazonaws.com/datasets/AdultIncomeBinaryClassification/test_data.csv')

label = 'class'
print(f'Train shape: {train_data.shape}')
print(f'Test shape:  {test_data.shape}')
train_data.head()

In [ ]:
# Examine class distribution
print(train_data[label].value_counts())
print('\nDtypes:')
print(train_data.dtypes)

## 2. Train the Predictor

Key parameters:
- **`label`** — the column you want to predict
- **`eval_metric`** — what to optimise; defaults to `'roc_auc'` for binary classification
- **`presets`** — trade-off between speed and accuracy (`'medium_quality'`, `'good_quality'`, `'high_quality'`, `'best_quality'`)
- **`time_limit`** — hard wall-clock budget in seconds

> **Tip:** Always set `time_limit`. Without it, `best_quality` will run until every model finishes, which can take hours on large datasets.

In [ ]:
predictor = TabularPredictor(
    label=label,
    eval_metric='roc_auc',
    path='./ag_tabular_model',   # where models are saved; set to None to use a temp dir
)

predictor.fit(
    train_data=train_data,
    time_limit=120,              # 2 minutes — increase for better results
    presets='medium_quality',
    verbosity=1,
)

## 3. Leaderboard — Which Model Won?

The leaderboard ranks all trained models by validation performance. The top row is the model AutoGluon will use when you call `.predict()` (usually the weighted ensemble).

In [ ]:
leaderboard = predictor.leaderboard(silent=True)
leaderboard[['model', 'score_val', 'fit_time', 'pred_time_val']].head(10)

## 4. Predict on Test Data

In [ ]:
# Hard class predictions
y_pred = predictor.predict(test_data)
print('Predicted labels (first 5):', y_pred[:5].tolist())

# Probability estimates for each class
y_prob = predictor.predict_proba(test_data)
print('\nProbability columns:', y_prob.columns.tolist())
y_prob.head()

In [ ]:
# Evaluate against held-out ground truth
# The test_data contains the label column — AutoGluon ignores it during prediction
# but uses it here to compute the metric
perf = predictor.evaluate(test_data, silent=True)
print('Test performance:', perf)

## 5. Feature Importance

AutoGluon computes permutation-based feature importance: it shuffles each feature and measures the drop in score. This requires a dataset with labels, so pass `test_data` (which still has the `class` column).

> **Note:** This can be slow because it re-runs predictions multiple times.

In [ ]:
importance = predictor.feature_importance(test_data, silent=True)
importance.head(10)

## 6. Quick Regression Demo

The API is identical for regression — AutoGluon infers `problem_type='regression'` from the numeric label.

In [ ]:
from sklearn.datasets import fetch_california_housing
import pandas as pd

housing = fetch_california_housing(as_frame=True)
df = housing.frame

split = int(len(df) * 0.8)
train_reg = df.iloc[:split]
test_reg  = df.iloc[split:]

reg_predictor = TabularPredictor(
    label='MedHouseVal',
    eval_metric='rmse',
    path='./ag_regression_model',
)
reg_predictor.fit(train_reg, time_limit=60, presets='medium_quality', verbosity=1)

print('\nRegression test RMSE:', reg_predictor.evaluate(test_reg, silent=True))

## 7. Saving & Loading a Predictor

Models are saved to the `path` you specified. Loading is straightforward:

In [ ]:
loaded = TabularPredictor.load('./ag_tabular_model')
print('Loaded predictor problem type:', loaded.problem_type)
print('Best model:', loaded.get_model_best())

## ⚠️ Practical Gotchas

| Gotcha | What Happens | Fix |
|--------|-------------|-----|
| **Models from older AutoGluon can't be loaded** | `TypeError` or `AttributeError` on `.load()` | Retrain the model with the current version |
| **Columns in test data don't match train** | Silent drop of extra columns; error if required column is missing | Keep column names identical (except the label) |
| **Label column in test data** | AutoGluon ignores it for `.predict()`, uses it for `.evaluate()` | No action needed — it is safe to pass test data with the label |
| **`predict_proba` column order** | Columns are label classes, not necessarily `[0, 1]` | Index with `y_prob[' >50K']` not `y_prob.iloc[:, 1]` |
| **Numeric column detected as categorical** | Happens when a column has only a few unique integers | Cast to `float` with `df['col'] = df['col'].astype(float)` before fitting |
| **Missing values** | Handled automatically — do not impute before fitting | Leave NaNs as-is; imputing can hurt performance |
| **`presets='best_quality'` without `time_limit`** | Training runs for many hours | Always set `time_limit` unless you know you have enough time |